## 3D Gaussian Splatting for Robot Navigation

### Introduction

Traditional robotics mapping produces sparse point clouds (ORB-SLAM) or 2D occupancy grids (SLAM Toolbox). Neither captures visual appearance. A robot with an occupancy grid of a kitchen knows where the walls are but cannot answer *"what does the kitchen look like from the doorway?"* or *"is this the same kitchen I saw before?"*

[3D Gaussian Splatting](https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/) (3DGS) produces a dense, renderable 3D representation by fitting a collection of 3D Gaussians to a set of posed images. Each Gaussian has a position, covariance (shape), color (via spherical harmonics), and opacity. Rendering is done by splatting these Gaussians onto an image plane — no ray marching needed — enabling real-time (100+ FPS) novel view synthesis.

For robotics, this enables:

- **Visual re-localization** — render what the robot expects to see from a candidate pose, compare with what it actually sees
- **Scene understanding** — with language-embedded splats, the 3D map becomes queryable: *"where is the couch?"* returns a rendered view and a pose
- **Navigation planning** — dense 3D geometry from splats generates richer obstacle representations than sparse point clouds

In this project you will use [gsplat](https://github.com/nerfstudio-project/gsplat), the actively maintained successor library from the Nerfstudio team. gsplat is a high-performance, modular library for differentiable Gaussian splatting — it provides the core rasterization engine, training strategies, and a `simple_trainer` example that serves as a complete training pipeline. Compared to the legacy Nerfstudio wrapper, gsplat is lighter, faster (uses ~4x less memory), and gives you direct access to the underlying primitives.

**An important note on simulation vs. reality.** In Part 2 of this project you will train a Gaussian Splat from images captured by a simulated D435i camera in the Gazebo house world. 3DGS faithfully reconstructs whatever it sees — if the input images are synthetic, the output splat will look synthetic too. A Gaussian Splat cannot make Gazebo renders more photorealistic than they already are. What 3DGS *does* give you, even from synthetic input, is a dense, view-consistent 3D representation that supports novel view synthesis, depth rendering, and semantic queries — capabilities that occupancy grids and sparse point clouds lack.

This works because 3DGS requires **multi-view consistency and accurate camera poses**, not photorealism. Gazebo satisfies both: its deterministic renderer produces perfectly consistent images across viewpoints, and simulated odometry provides ground-truth poses with zero drift. The original 3DGS paper evaluated on the synthetic Blender/NeRF-Synthetic dataset, and all major GS-SLAM systems ([SplaTAM](https://spla-tam.github.io/), [GS-SLAM](https://gs-slam.github.io/), [RTG-SLAM](https://arxiv.org/abs/2404.19706)) benchmark on [Replica](https://wijmans.xyz/publication/replica-2019/) — a synthetic indoor dataset rendered with OpenGL. That said, Gazebo's default rendering quality (Ogre2) is below Replica's level of fidelity, so expect the reconstruction to reflect the visual quality of Gazebo's house world, not a photorealistic scan. The sim-to-real comparison in Task 2.4 is designed to surface exactly this gap.

Gazebo's rendering pipeline does support higher-fidelity materials ([PBR with normal/roughness/metalness maps](https://gazebosim.org/api/rendering/10/classgz_1_1rendering_1_1Ogre2Material.html), [baked light maps with global illumination](https://gazebosim.org/api/rendering/9/lightmap.html), and [configurable sensor noise](https://gazebosim.org/libs/sensors/)), but the default house world does not fully exploit these capabilities.

This project has three parts:

| Part | Focus | Environment |
|------|-------|----------|
| **Part 0** | Run a working gsplat demo end-to-end | Mip-NeRF 360 garden scene (provided) |
| **Part 1** | Capture and train your first Gaussian Splat | Your own room (any camera) |
| **Part 2** | Integrate GS capture into the ROS pipeline | Gazebo simulation (TurtleBot + D435i) |

### Prerequisites

- Completed [Camera Calibration](/aiml-common/lectures/sensor-models/cameras/camera-calibration) lecture
- A camera for scene capture — **any** of the following works:
  - iPhone 12 Pro or newer (LiDAR) with [Record3D](https://record3d.app/) — best quality, no COLMAP needed
  - Any iPhone/Android with [Polycam](https://poly.cam/) or [Scaniverse](https://scaniverse.com/)
  - Laptop webcam or USB camera — use COLMAP for pose estimation (see below)
- Workstation with NVIDIA GPU (RTX 3060+ for preview quality, RTX 3090/4090 for full quality)
- Docker with NVIDIA Container Toolkit installed

### COLMAP Installation

[COLMAP](https://colmap.github.io/) is a Structure-from-Motion (SfM) and Multi-View Stereo (MVS) pipeline that estimates camera poses from unposed images. It is required if you are using a monocular camera (laptop webcam, USB camera, or phone without LiDAR).

Unlike the legacy Nerfstudio workflow (which wrapped COLMAP behind `ns-process-data`), with gsplat you run COLMAP directly. gsplat's dataset parser expects raw COLMAP output (`sparse/0/cameras.bin`, `images.bin`, `points3D.bin`).

**Option A — Inside the gsplat Docker container:**

COLMAP can be installed via `apt` inside the container:

```bash
apt-get update && apt-get install -y colmap
```

Then run the COLMAP pipeline directly:

```bash
# Feature extraction
colmap feature_extractor --database_path db.db --image_path images/
# Feature matching
colmap exhaustive_matcher --database_path db.db
# Sparse reconstruction
mkdir -p sparse/0
colmap mapper --database_path db.db --image_path images/ --output_path sparse/
```

**Option B — Native installation on Ubuntu:**

```bash
sudo apt-get install colmap
```

For GPU-accelerated feature matching (significantly faster on large image sets), build from source with CUDA support. Follow the [official build instructions](https://colmap.github.io/install.html).

**Option C — pip (CPU-only, slower but simplest):**

```bash
pip install pycolmap
```

### Background Reading

Before starting, study these resources in order:

1. **Conceptual overview:** [Introduction to 3D Gaussian Splatting](https://huggingface.co/blog/gaussian-splatting) (Hugging Face blog) — accessible explanation of the theory
2. **Original paper:** [3D Gaussian Splatting for Real-Time Radiance Field Rendering](https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/) (SIGGRAPH 2023) — the foundational work
3. **Robotics survey:** [3DGS in Robotics: A Survey](https://arxiv.org/abs/2410.12262) — how 3DGS is being used for SLAM, navigation, and manipulation
4. **gsplat documentation and examples:** [github.com/nerfstudio-project/gsplat/tree/main/examples](https://github.com/nerfstudio-project/gsplat/tree/main/examples) — the training framework you will use, including `simple_trainer.py` and dataset parsers

### Compute Requirements

| Task | VRAM | GPU | Time |
|------|------|-----|------|
| Training gsplat, 7K steps (preview) | 4-6 GB | RTX 3060+ | ~5 min |
| Training gsplat, 30K steps (full) | 6-8 GB | RTX 3090/4090 | ~15-20 min |
| Rendering trained splat | — | Any CUDA GPU | 100-200+ FPS |
| COLMAP SfM (200 images) | 2-4 GB | CPU or CUDA GPU | 5-30 min |

---

## Part 0: Working Demo

Before capturing your own data, run this end-to-end demo to verify your environment and see gsplat in action. It downloads the **garden** scene from the [Mip-NeRF 360](https://huggingface.co/datasets/mileleap/mipnerf360) benchmark dataset, trains a Gaussian splat, and renders a fly-through video — all in a few minutes on a single GPU.

This gives you a working baseline to compare against when you train on your own captures in Parts 1 and 2.

The demo runs on **Google Colab** (T4/A100) or your local **Docker GPU** environment.

In [ ]:
# Environment detection
import sys, os, shutil

IN_COLAB = "google.colab" in sys.modules
WORKDIR = "/content" if IN_COLAB else os.getcwd()
DATA_ROOT = os.path.join("/content" if IN_COLAB else os.path.expanduser("~"), "data")
RESULT_DIR = os.path.join(WORKDIR, "results")
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

print(f"Environment: {'Colab' if IN_COLAB else 'Docker/Local'}")
print(f"Working directory: {WORKDIR}")
print(f"Data root: {DATA_ROOT}")

In [ ]:
# Install dependencies
import sys, subprocess, shutil

IN_COLAB = "google.colab" in sys.modules

def pip_install(*packages):
    """Install packages using uv (if available) or pip."""
    if shutil.which("uv"):
        subprocess.check_call(["uv", "pip", "install", "-q"] + list(packages))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(packages))

if IN_COLAB:
    pip_install("torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu121")

# Install gsplat if not already available
try:
    import gsplat
    print(f"gsplat {gsplat.__version__} already installed")
except ImportError:
    print("Installing gsplat...")
    pip_install("gsplat")

import torch
print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone gsplat examples and install dependencies
import subprocess, sys, shutil

GSPLAT_DIR = os.path.join(WORKDIR, "gsplat")

if not os.path.exists(GSPLAT_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1", "--recurse-submodules", "--branch", "v1.5.3",
        "https://github.com/nerfstudio-project/gsplat.git", GSPLAT_DIR])
else:
    print(f"gsplat repo already cloned at {GSPLAT_DIR}")

def uv_or_pip(*args, no_build_isolation=False):
    if shutil.which("uv"):
        cmd = ["uv", "pip", "install", "-q"] + list(args)
        if no_build_isolation:
            cmd.insert(4, "--no-build-isolation")
        subprocess.check_call(cmd)
    else:
        cmd = [sys.executable, "-m", "pip", "install", "-q"] + list(args)
        if no_build_isolation:
            cmd.insert(5, "--no-build-isolation")
        subprocess.check_call(cmd)

print("Installing gsplat example dependencies...")

# PyPI packages
uv_or_pip(
    "numpy<2.0.0", "scipy", "scikit-learn", "opencv-python", "Pillow", "piexif",
    "imageio[ffmpeg]", "tensorboard", "torchmetrics[image]", "tensorly",
    "tyro>=0.8.8", "pyyaml", "tqdm", "matplotlib", "splines", "viser",
)

# Git packages that need torch at build time (--no-build-isolation)
uv_or_pip("git+https://github.com/rmbrualla/pycolmap@cc7ea4b7301720ac29287dbe450952511b32125e")
uv_or_pip("git+https://github.com/nerfstudio-project/nerfview@4538024fe0d15fd1a0e4d760f3695fc44ca72787")
uv_or_pip("git+https://github.com/rahul-goel/fused-ssim@328dc9836f513d00c4b5bc38fe30478b4435cbb5",
           no_build_isolation=True)
uv_or_pip("git+https://github.com/harry7557558/fused-bilagrid@49f0ef06c9f81810fb9b5dd9027cf1844950cc16",
           no_build_isolation=True)

# Verify
trainer_path = os.path.join(GSPLAT_DIR, "examples", "simple_trainer.py")
assert os.path.exists(trainer_path), f"simple_trainer.py not found at {trainer_path}"
print(f"Trainer: {trainer_path}")

import tyro
print(f"tyro {tyro.__version__} OK")

In [ ]:
# Download the garden scene from Mip-NeRF 360 dataset
#
# gsplat expects COLMAP format:
#   <data_dir>/sparse/0/{cameras.bin, images.bin, points3D.bin}
#   <data_dir>/images/      (full-res)
#   <data_dir>/images_4/    (4x downsampled, optional)

from huggingface_hub import snapshot_download

SCENE = "garden"  # other options: bicycle, bonsai, counter, kitchen, room, stump

scene_dir = os.path.join(DATA_ROOT, "360_v2", SCENE)

if not os.path.exists(os.path.join(scene_dir, "sparse")):
    snapshot_download(
        repo_id="mileleap/mipnerf360",
        repo_type="dataset",
        allow_patterns=f"{SCENE}/**",
        local_dir=os.path.join(DATA_ROOT, "360_v2"),
    )
    print(f"Downloaded scene '{SCENE}'")
else:
    print(f"Scene '{SCENE}' already downloaded")

# Verify COLMAP files
sparse_dir = os.path.join(scene_dir, "sparse", "0")
for f in ["cameras.bin", "images.bin", "points3D.bin"]:
    path = os.path.join(sparse_dir, f)
    assert os.path.exists(path), f"Missing {path}"
print(f"COLMAP files verified in {sparse_dir}")

# Show image count
images_dir = os.path.join(scene_dir, "images")
n_images = len([f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.png'))])
print(f"Images: {n_images}")

In [ ]:
# Train the Gaussian splat model
#
# In Docker/headless: train for a limited number of steps.
# In Colab: train for more steps (adjust as needed for your GPU).

MAX_STEPS = 2000 if not IN_COLAB else 7000
DATA_FACTOR = 4  # use 4x downsampled images

result_dir = os.path.join(RESULT_DIR, SCENE)

train_cmd = (
    f"cd {GSPLAT_DIR}/examples && "
    f"python simple_trainer.py default"
    f" --data_dir {scene_dir}"
    f" --data_factor {DATA_FACTOR}"
    f" --result_dir {result_dir}"
    f" --max_steps {MAX_STEPS}"
    f" --save_steps {MAX_STEPS}"
    f" --eval_steps {MAX_STEPS}"
    f" --disable_viewer"
)

print(f"Training for {MAX_STEPS} steps with data_factor={DATA_FACTOR}")
print(f"Command: {train_cmd}")
!{train_cmd}

In [ ]:
# Render a fly-through video from the trained model
import glob

# Find the latest checkpoint
ckpt_pattern = os.path.join(result_dir, "ckpts", "ckpt_*.pt")
ckpts = sorted(glob.glob(ckpt_pattern))

if not ckpts:
    print("No checkpoints found. Complete training first.")
else:
    latest_ckpt = ckpts[-1]
    print(f"Using checkpoint: {latest_ckpt}")

    render_cmd = (
        f"cd {GSPLAT_DIR}/examples && "
        f"python simple_trainer.py default"
        f" --data_dir {scene_dir}"
        f" --data_factor {DATA_FACTOR}"
        f" --result_dir {result_dir}"
        f" --ckpt {latest_ckpt}"
        f" --render_traj_path ellipse"
        f" --disable_viewer"
    )

    print(f"Rendering ellipse trajectory...")
    !{render_cmd}

    # Check for rendered video
    videos = glob.glob(os.path.join(result_dir, "**", "*.mp4"), recursive=True)
    if videos:
        print(f"\nRendered video: {videos[0]}")
    else:
        print("No video output found — check render logs above.")

In [ ]:
# Display the rendered video (if available)
from IPython.display import Video, display

videos = glob.glob(os.path.join(result_dir, "**", "*.mp4"), recursive=True)
if videos:
    display(Video(videos[0], embed=True, width=640))
else:
    print("No rendered video to display.")

---

## Part 1: Capture and Train Your First Gaussian Splat

In this part you will capture a real-world scene (your room, living space, or lab) using any available camera and train a Gaussian Splat from the captured data. The goal is to build intuition for how capture quality affects reconstruction quality before moving to the robotic pipeline.

### Task 1.1: Scene Capture

Choose a room-scale scene (3-6 meters across). Good scenes have varied geometry and texture — avoid blank white walls. Follow these capture best practices regardless of camera type:

- Walk **slowly** — fast motion causes blur and tracking loss
- Maintain **70-80% overlap** between consecutive frames
- Capture from **multiple heights** (standing, crouching) to cover vertical surfaces
- Cover the full scene — gaps in coverage become holes in reconstruction
- Aim for **200-500 frames** for a single room

#### Path A: iPhone with LiDAR (best quality)

If you have an iPhone 12 Pro or newer:

1. Install [Record3D](https://record3d.app/) from the App Store.
2. Record a walkthrough of the scene.
3. Export as **"EXR + JPG sequence"**.
4. Also export **"Zipped PLY point clouds"** — these provide a LiDAR-based point cloud that dramatically improves Gaussian initialization.
5. Transfer both exports to your workstation.

Processing: Export the images from Record3D and run COLMAP on them to produce the sparse reconstruction that gsplat expects. If Record3D provides a COLMAP-compatible export, use that directly. Otherwise:

```bash
cd /data/captures/my_room_r3d

# Feature extraction
colmap feature_extractor \
  --database_path db.db \
  --image_path images/ \
  --ImageReader.single_camera 1

# Feature matching
colmap exhaustive_matcher --database_path db.db

# Sparse reconstruction
mkdir -p sparse/0
colmap mapper \
  --database_path db.db \
  --image_path images/ \
  --output_path sparse/
```

If Record3D provides ARKit poses, you can alternatively convert them to COLMAP format (write `cameras.txt`, `images.txt`, `points3D.txt` in `sparse/0/`) to skip the SfM step entirely.

#### Path B: Any phone with Polycam/Scaniverse

If you have any iPhone or Android phone:

1. Install [Polycam](https://poly.cam/) or [Scaniverse](https://scaniverse.com/).
2. Use the app's photo mode to capture the scene (follow the overlap guidelines above).
3. Export the images and transfer to your workstation.

Processing — run COLMAP on the exported images:

```bash
cd /data/captures/my_room_polycam

colmap feature_extractor \
  --database_path db.db \
  --image_path images/ \
  --ImageReader.single_camera 1

colmap exhaustive_matcher --database_path db.db

mkdir -p sparse/0
colmap mapper \
  --database_path db.db \
  --image_path images/ \
  --output_path sparse/
```

#### Path C: Laptop webcam or USB camera (COLMAP)

If you only have a laptop webcam or USB camera:

1. **Capture images:** Record a video of your room, then extract frames, or take individual photos while walking around the scene. For video extraction:
   ```bash
   # Extract 1 frame per second from video (adjust -r for density)
   ffmpeg -i my_room_video.mp4 -r 1 -q:v 2 /data/captures/my_room_images/frame_%04d.jpg
   ```
   Alternatively, write a short Python script using OpenCV to capture frames from the webcam at regular intervals as you walk around the room.

2. **Use your calibration:** If you completed the Camera Calibration assignment, you can provide your camera intrinsics to improve COLMAP's accuracy. Otherwise, COLMAP will estimate intrinsics automatically (less accurate, especially for wide-angle laptop cameras with significant distortion).

3. **Process with COLMAP:**
   ```bash
   cd /data/captures/my_room_images

   colmap feature_extractor \
     --database_path db.db \
     --image_path images/ \
     --ImageReader.single_camera 1

   colmap exhaustive_matcher --database_path db.db

   mkdir -p sparse/0
   colmap mapper \
     --database_path db.db \
     --image_path images/ \
     --output_path sparse/
   ```
   This runs the full COLMAP SfM pipeline: feature extraction (SIFT), feature matching, sparse reconstruction, and undistortion. Expect 5-30 minutes depending on image count and whether GPU-accelerated matching is available.

4. **Troubleshooting COLMAP failures:** If COLMAP fails to reconstruct (common causes: insufficient overlap, textureless surfaces, motion blur):
   - Check that most images were registered — COLMAP will report the number of registered images in the reconstruction
   - Re-capture with slower movement and more overlap
   - Try `--SiftExtraction.use_gpu 1` for faster feature extraction if you have a CUDA GPU

**Comparison of capture paths:**

| Path | Poses from | Depth available | Point cloud init | Quality |
|------|-----------|----------------|-----------------|--------|
| A: Record3D (LiDAR) | ARKit (or COLMAP) | Yes (LiDAR) | Yes (PLY export) | Best |
| B: Polycam/Scaniverse | COLMAP | Sometimes | SfM sparse points | Good |
| C: Webcam + COLMAP | COLMAP SfM | No | SfM sparse points | Adequate |

Path C produces RGB-only splats without depth supervision. This is perfectly valid — the original 3DGS paper used RGB-only with COLMAP poses. Initialization will use COLMAP's sparse point cloud.

### Task 1.2: gsplat Setup

You need a working gsplat environment with GPU support. Choose one of the following options:

#### Option A: gsplat Docker container (recommended)

Use the eng-ai-agents docker-compose service:

```bash
docker compose run --rm gsplat.dev.gpu bash
```

Or run a standalone container:

```bash
docker run --gpus all \
  -v $(pwd)/data:/data \
  -p 8080:8080 \
  --rm -it \
  --shm-size=12gb \
  pytorch/pytorch:2.7.1-cuda12.8-cudnn9-devel bash

# Inside the container:
pip install gsplat
git clone https://github.com/nerfstudio-project/gsplat.git /opt/gsplat
cd /opt/gsplat && pip install -r examples/requirements.txt
```

#### Option B: Native pip install

If you prefer a local installation without Docker:

```bash
# Create a conda environment
conda create -n gsplat python=3.10 -y
conda activate gsplat

# Install PyTorch with CUDA (check https://pytorch.org for your CUDA version)
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

# Install gsplat
pip install gsplat

# Clone gsplat for examples and dataset parsers
git clone https://github.com/nerfstudio-project/gsplat.git
cd gsplat && pip install -r examples/requirements.txt

# Verify
python -c "import gsplat; print(gsplat.__version__)"
```

### Task 1.3: Train Your First Splat

Inside your gsplat environment (Docker or native), train on your processed data using the `simple_trainer` example:

```bash
cd /opt/gsplat  # or wherever you cloned gsplat

python -m examples.simple_trainer default \
  --data_dir /data/captures/my_room_processed \
  --data_factor 4 \
  --result_dir /data/results/my_room \
  --max_steps 30000
```

The gsplat `simple_trainer` includes a built-in [viser](https://viser.studio/) viewer that opens automatically during training. Navigate to `localhost:8080` in your browser to watch training progress in real time — you can orbit the scene, zoom in, and observe Gaussians being refined as training proceeds.

Key `simple_trainer` arguments:

| Argument | Description | Default |
|----------|-------------|---------|
| `--data_dir` | Path to COLMAP dataset (must contain `sparse/0/` and `images/`) | required |
| `--data_factor` | Image downscale factor (4 = quarter resolution, faster training) | 1 |
| `--result_dir` | Where to save checkpoints, PLY files, and metrics | required |
| `--max_steps` | Total training iterations | 30000 |
| `--eval_steps` | Evaluate every N steps | [7000, 30000] |
| `--save_steps` | Save checkpoint every N steps | [7000, 30000] |

The first positional argument selects the training strategy: `default` (DefaultStrategy) or `mcmc` (MCMCStrategy). Start with `default`.

### Task 1.4: Evaluate and Experiment

Once your first splat is trained, review the evaluation results and begin experimenting:

1. **Evaluate** your trained splat. The `simple_trainer` evaluates automatically at steps specified by `--eval_steps`. Results (PSNR, SSIM, LPIPS) are printed to the console and saved in the `result_dir`. Report these metrics.

2. **Export PLY:** The `simple_trainer` saves `.ply` files automatically at steps specified by `--save_steps`. Check your `result_dir` for `*.ply` files. These can be loaded into any 3D viewer (MeshLab, CloudCompare, or the [online SuperSplat viewer](https://playcanvas.com/supersplat/editor)).

3. **Experiment** with at least two of the following variations and compare results:

| Experiment | What to vary | What to measure | Notes |
|---|---|---|---|
| Training duration | 7K vs 15K vs 30K steps | PSNR/SSIM, training time, visual quality | All paths |
| Training strategy | `default` (DefaultStrategy) vs `mcmc` (MCMCStrategy) | Quality metrics, VRAM usage, training time | `mcmc` does not need densification heuristics |
| Init type | `--init_type sfm` vs `--init_type random` | Convergence speed, final quality | `sfm` uses COLMAP sparse points; `random` starts from scratch |
| Quality tuning | `--opacity_reg 0.01 --scale_reg 0.01` vs defaults | Floaters, edge quality | Regularization reduces artifacts |
| Data factor | `--data_factor 1` (full res) vs `--data_factor 4` (quarter) vs `--data_factor 8` | Quality vs training time and memory | Higher factor = faster but lower quality |
| Capture density | Vary frame extraction rate (e.g., 1 FPS vs 3 FPS vs 5 FPS) | Quality vs training time | Path C (video extraction) |

4. **Capture a second scene** — a different room or area — and train a splat. You will use both scenes to discuss coverage vs. quality tradeoffs in your report.

### Deliverables for Part 1

- Screenshots or screen recordings of the viser viewer showing your trained splats
- Evaluation metrics (PSNR, SSIM, LPIPS) for each experiment variation
- Written analysis: what capture choices and training configurations affected quality and why

---

## Part 2: Gaussian Splatting in the ROS Pipeline

In Part 1 you captured real scenes with a physical camera. Now you will integrate Gaussian Splatting into the TurtleBot3 simulation pipeline, where RGB-D images and poses come from Gazebo via ROS 2 and Zenoh.

The resulting splat will be a faithful 3D reconstruction of the Gazebo house world — it will look as realistic (or as synthetic) as the Gazebo renderer itself. The value is not photorealism but the *representation*: a dense, renderable, queryable 3D map that an occupancy grid or sparse point cloud cannot provide. Gazebo's deterministic renderer and ground-truth odometry give you ideal conditions for 3DGS (perfect multi-view consistency, zero pose drift), letting you focus on the capture pipeline and exploration strategy rather than sensor noise.

### Architecture Overview

The capture pipeline follows the same Zenoh subscriber pattern as the existing `object_detector.py`:

```
Gazebo House World (D435i sim)
    |
    | ros_gz_bridge -> zenoh-bridge-ros2dds
    v
gs_capture.py (Zenoh subscriber)
    |
    | Keyframe gating (distance + angle)
    | ROS -> COLMAP coordinate transform
    | Depth float32 m -> kept as-is (gsplat handles float32)
    v
data/captures/house_run/
    +-- sparse/0/
    |   +-- cameras.bin or cameras.txt
    |   +-- images.bin or images.txt
    |   +-- points3D.bin or points3D.txt
    +-- images/           (RGB PNGs/JPGs)
    +-- images_4/         (optional: 4x downsampled)
    |
    v
python -m examples.simple_trainer default --data_dir <path>
    |
    v
Trained Gaussian Splat of the House World
```

### Task 2.1: Implement `gs_capture.py`

Write a Zenoh capture script (`detector/gs_capture.py`) that subscribes to the robot's RGB, depth, and odometry streams and writes keyframes in COLMAP format.

**Data sources (Zenoh keys):**

| Zenoh Key | ROS Topic | Data | Frame |
|---|---|---|---|
| `camera/color/image_raw` | `/camera/color/image_raw` | RGB image (sensor_msgs/Image, CDR) | camera_optical_frame |
| `camera/depth/image_rect_raw` | `/camera/depth/image_rect_raw` | Depth image (float32 meters, CDR) | camera_depth_frame |
| `odom` | `/odom` | Robot odometry (nav_msgs/Odometry, CDR) | odom -> base_link |

**Key design decisions you must implement:**

1. **Frame synchronization:** Capture is RGB-triggered. When an RGB frame arrives, pair it with the most recent depth and odom. Drop the frame if depth or odom is more than 200 ms stale.

2. **Keyframe gating:** Same logic as `object_detector.py` — only capture when the robot has moved more than a distance threshold or rotated more than an angle threshold since the last keyframe. Expose these as CLI arguments (defaults: 0.3 m, 10 degrees).

3. **Coordinate transform:** This is the most critical part. gsplat's COLMAP parser expects camera poses in the **COLMAP convention**, where poses are stored as **world-to-camera** transforms (the inverse of what ROS provides).

   The transform chain is:
   ```
   T_world_camera = T_odom_baselink @ T_baselink_camera
   ```

   Then convert to COLMAP convention (world-to-camera):
   ```python
   # T_world_camera is camera-to-world (4x4)
   R_c2w = T_world_camera[:3, :3]
   t_c2w = T_world_camera[:3, 3]

   # COLMAP stores world-to-camera:
   R_w2c = R_c2w.T
   t_w2c = -R_w2c @ t_c2w

   # Convert rotation to quaternion (COLMAP order: QW, QX, QY, QZ)
   from scipy.spatial.transform import Rotation
   quat = Rotation.from_matrix(R_w2c).as_quat()  # returns [x, y, z, w]
   qw, qx, qy, qz = quat[3], quat[0], quat[1], quat[2]
   ```

   Where:
   - `T_odom_baselink` — from the `/odom` message (position + quaternion -> 4x4 matrix)
   - `T_baselink_camera` — static offset from URDF: translation `[0.064, -0.065, 0.094]`, no rotation

   Note: Unlike the Nerfstudio pipeline, there is **no need** for the `T_ros_optical_to_nerfstudio` 180-degree flip around the x-axis. COLMAP and the ROS optical frame both use the convention of z-forward, x-right, y-down.

4. **Depth handling:** gsplat's COLMAP parser does not use depth files directly. Depth supervision is not part of the default `simple_trainer` pipeline. You may optionally store depth images for future use, but they are not required for training.

5. **Missing odom handling:** If no odom has been received, capture pauses. Frames without valid poses are never written. Log a warning if odom is missing for more than 10 seconds.

6. **Output format:** Write COLMAP text files instead of `transforms.json`:

   **`sparse/0/cameras.txt`** — single camera entry with the PINHOLE model:
   ```
   # Camera list with one line of data per camera:
   #   CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[]
   # Number of cameras: 1
   1 PINHOLE 320 240 277.13 277.13 160.0 120.0
   ```

   **`sparse/0/images.txt`** — one entry per keyframe (two lines per image: header + empty points2D line):
   ```
   # Image list with two lines of data per image:
   #   IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME
   #   POINTS2D[] as (X, Y, POINT3D_ID)
   1 0.9239 0.0 0.0 0.3827 1.234 0.567 0.890 1 frame_000001.png
   
   2 0.8660 0.0 0.0 0.5000 2.345 0.678 0.901 1 frame_000002.png
   
   ```

   **`sparse/0/points3D.txt`** — can be empty initially (gsplat can initialize with random points via `--init_type random`):
   ```
   # 3D point list with one line of data per point:
   #   POINT3D_ID, X, Y, Z, R, G, B, ERROR, TRACK[] as (IMAGE_ID, POINT2D_IDX)
   ```

**Reference:** Study `object_detector.py` for the Zenoh subscription pattern, CDR deserialization, and keyframe gating logic. Your capture script follows the same structure but writes a different output format.

### Task 2.2: Capture and Train from Simulation

Use the capture script to collect data from the Gazebo house world:

```bash
# 1. Launch the simulation stack
docker compose up -d demo-world-house zenoh-router zenoh-bridge gs-capture

# 2. Drive the robot through the house
#    Use teleop or Nav2 waypoints to explore the environment
#    Aim for systematic coverage — visit every room

# 3. Stop capture
docker compose stop gs-capture

# 4. Train (inside gsplat container)
cd /opt/gsplat
python -m examples.simple_trainer default \
  --data_dir /data/captures/house_run \
  --data_factor 1 \
  --result_dir /data/results/house_run \
  --max_steps 30000

# 5. View with viser
# The viewer opens automatically during training at localhost:8080
```

### Task 2.3: Exploration Strategy Experiments

The quality of a Gaussian Splat depends heavily on how well the scene was covered during capture. Run at least two of these experiments:

| Experiment | Description |
|---|---|
| **Teleop vs Nav2 waypoints** | Compare manual exploration with pre-planned waypoint sequences. Which gives better coverage? |
| **Dense vs sparse capture** | Compare `--keyframe-dist 0.1 --keyframe-angle 5` with `--keyframe-dist 1.0 --keyframe-angle 30`. How does frame count affect quality and training time? |
| **Partial vs full coverage** | Train a splat from a single room capture vs. the full house. Where do reconstruction holes appear? |
| **Init type** | Compare `--init_type sfm` (using COLMAP sparse points from `points3D.txt`) vs `--init_type random` (no prior geometry). How much does SfM initialization help when starting from simulation poses? |

### Task 2.4: Sim-to-Real Comparison

Compare your simulation splat (Part 2) with your real-world splat (Part 1). This comparison is designed to make the sim-real gap concrete and measurable.

1. Render 5 novel views from random poses in each trained splat.
2. Qualitatively compare: texture detail, geometric accuracy, lighting, artifacts.
3. Discuss:
   - What are the fundamental differences between simulated and real-world captures?
   - Where does the Gazebo splat faithfully reproduce geometry but fail on visual realism?
   - What would need to change in the Gazebo world (materials, lighting, sensor noise) to narrow the gap?

### Deliverables for Part 2

- Source code for `gs_capture.py` with inline comments explaining the coordinate transform
- A unit test that verifies the coordinate transform: given a known ROS pose, assert the expected COLMAP quaternion and translation
- Evaluation metrics and screenshots for each exploration experiment
- Written sim-to-real comparison analysis

---

## References

### Foundational

- [3D Gaussian Splatting for Real-Time Radiance Field Rendering](https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/) — SIGGRAPH 2023
- [3DGS in Robotics: A Survey](https://arxiv.org/abs/2410.12262) — comprehensive survey of 3DGS applications in robotics
- [Awesome 3DGS SLAM](https://github.com/KwanWaiPang/Awesome-3DGS-SLAM) — curated list of GS-SLAM papers

### GS-SLAM Systems

- [RTG-SLAM](https://arxiv.org/abs/2404.19706) — real-time RGB-D Gaussian SLAM (SIGGRAPH 2024)
- [SplaTAM](https://spla-tam.github.io/) — best reconstruction quality for RGB-D (CVPR 2024)
- [Photo-SLAM](https://arxiv.org/abs/2311.16728) — runs on Jetson AGX Orin (CVPR 2024)

### Re-localization

- [Splat-Loc](https://arxiv.org/abs/2312.02126) — render-and-compare re-localization at ~25 Hz
- [GS-Loc](https://arxiv.org/abs/2410.06165) — vision foundation model-driven (RAL 2025)

### Frameworks and Tools

- [gsplat](https://github.com/nerfstudio-project/gsplat) — high-performance Gaussian splatting library, actively maintained successor from the Nerfstudio team
- [COLMAP](https://colmap.github.io/) — Structure-from-Motion for camera pose estimation from unposed images ([install guide](https://colmap.github.io/install.html), [tutorial](https://colmap.github.io/tutorial.html))
- [ROSplat](https://github.com/shadygm/ROSplat) — ROS 2 Jazzy GS visualization
- [Record3D](https://record3d.app/) — iPhone LiDAR capture app
- [Polycam](https://poly.cam/) — alternative mobile capture
- [Scaniverse](https://scaniverse.com/) — cross-platform mobile capture (iPhone and Android)
- [viser](https://viser.studio/) — 3D visualization library used by gsplat's built-in viewer

---

## Summary of Deliverables

| Part | Deliverables |
|------|-------------|
| **Part 0** | Successful training run on the garden scene, rendered fly-through video, PSNR/SSIM metrics |
| **Part 1** | Trained splat of your room, evaluation metrics, experiment comparisons, written analysis |
| **Part 2** | `gs_capture.py` source + unit test, sim splat with exploration experiments, sim-to-real comparison |

All gsplat outputs should go to `/data/results/` (mounted from `./data/results` on the host). Always use `--result_dir /data/results/` to keep artifacts in the mounted volume.